# Run J: D-FINE-S, 640 px, one-class small-defect detection

This notebook reuses the exact `run_a_yolo_dataset` output used in the YOLO runs. It converts the existing YOLO bounding boxes to COCO JSON for D-FINE, trains D-FINE-S, then evaluates validation plus overall, small, medium, and large test sets.

Before running: attach the Kaggle notebook output that contains `run_a_yolo_dataset` as an input. Turn Internet on so Kaggle can clone the official D-FINE repository and install its requirements.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
from collections import Counter

RUN_NAME = 'RunJ_dfine_s_imgsz640'
IMG_SIZE = 640
BATCH_SIZE = 8
EPOCHS = 50
SEED = 42
NUM_WORKERS = 2
# D-FINE-S was numerically unstable at the official custom-config learning rate
# on this mixed-domain one-class dataset, so use a conservative fine-tuning rate.
BASE_LR = 0.0001
BACKBONE_LR = 0.00005

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
DFINE_REPO = WORKING_ROOT / 'D-FINE'
COCO_ROOT = WORKING_ROOT / 'dfine_defect_coco'
RUN_DIR = WORKING_ROOT / 'dfine_runs' / RUN_NAME
FINAL_OUTPUT_DIR = WORKING_ROOT / 'final_outputs' / RUN_NAME

assert IMG_SIZE == 640, 'This D-FINE config is deliberately fixed to 640 for a fair comparison.'
print({'run': RUN_NAME, 'imgsz': IMG_SIZE, 'batch': BATCH_SIZE, 'epochs': EPOCHS})

In [ ]:
# Kaggle Internet must be enabled for this cell.
if not DFINE_REPO.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/Peterande/D-FINE.git', str(DFINE_REPO)
    ], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(DFINE_REPO / 'requirements.txt'), 'pycocotools'
], check=True)

print('D-FINE ready at:', DFINE_REPO)

In [ ]:
# Accept either the earlier YOLO output or the usual SmallDefectPreprocessing input.
# The fallback reproduces the fixed 70/15/15 split from the preprocessing output.
import random
from collections import defaultdict

DATASET_NAMES = ['DAGM', 'GC10-DET', 'KolektorSDD2', 'MPDD', 'MTD', 'Severstal', 'VisA']
SIZE_BUCKETS = ['small', 'medium', 'large']
SPLIT_NAMES = ('train', 'val', 'test', 'test_small', 'test_medium', 'test_large')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

dataset_candidates = sorted(KAGGLE_INPUT_ROOT.glob('**/run_a_yolo_dataset'))
if dataset_candidates:
    YOLO_DATASET_DIR = dataset_candidates[0]
    print('Using existing prepared YOLO dataset:', YOLO_DATASET_DIR)
else:
    expected_datasets = set(DATASET_NAMES)
    source_candidates = []
    for root, dirs, _ in os.walk(KAGGLE_INPUT_ROOT):
        matches = expected_datasets.intersection(dirs)
        if len(matches) >= 5:
            source_candidates.append((len(matches), Path(root)))

    if not source_candidates:
        raise FileNotFoundError(
            'Could not find either run_a_yolo_dataset or the processed dataset folders. '
            'Attach SmallDefectPreprocessing as a Kaggle input.'
        )

    source_candidates.sort(key=lambda item: (-item[0], len(str(item[1]))))
    SOURCE_ROOT = source_candidates[0][1]
    YOLO_DATASET_DIR = WORKING_ROOT / 'run_a_yolo_dataset'
    print('Using SmallDefectPreprocessing source:', SOURCE_ROOT)

    def label_stem_candidates(image_stem):
        base = image_stem.removesuffix('_defect')
        return [image_stem, image_stem.replace('_defect', '_bbs'), base, base + '_bbs']

    samples = []
    for dataset_name in DATASET_NAMES:
        for size_bucket in SIZE_BUCKETS:
            image_dir = SOURCE_ROOT / dataset_name / size_bucket / 'images'
            label_dir = SOURCE_ROOT / dataset_name / size_bucket / 'labels_yolo'
            if not image_dir.exists() or not label_dir.exists():
                print('Missing source folder:', dataset_name, size_bucket)
                continue

            label_index = {path.stem: path for path in label_dir.glob('*.txt')}
            matched = 0
            for image_path in image_dir.iterdir():
                if image_path.suffix.lower() not in IMAGE_EXTS:
                    continue
                label_path = next((label_index[stem] for stem in label_stem_candidates(image_path.stem) if stem in label_index), None)
                if label_path is None:
                    continue
                samples.append({
                    'image_path': image_path, 'label_path': label_path,
                    'dataset': dataset_name, 'size': size_bucket,
                    'stratum': dataset_name + '_' + size_bucket,
                })
                matched += 1
            print(f'{dataset_name}/{size_bucket}: {matched} matched')

    if len(samples) != 12670:
        raise RuntimeError(f'Expected 12,670 usable image-label pairs, found {len(samples)}.')

    by_stratum = defaultdict(list)
    for sample in samples:
        by_stratum[sample['stratum']].append(sample)

    rng = random.Random(SEED)
    train_samples, val_samples, test_samples = [], [], []
    for _, group in sorted(by_stratum.items()):
        group = list(group)
        rng.shuffle(group)
        train_end = int(len(group) * 0.70)
        val_end = train_end + int(len(group) * 0.15)
        train_samples.extend(group[:train_end])
        val_samples.extend(group[train_end:val_end])
        test_samples.extend(group[val_end:])

    rng.shuffle(train_samples)
    rng.shuffle(val_samples)
    rng.shuffle(test_samples)

    export_groups = {
        'train': train_samples, 'val': val_samples, 'test': test_samples,
        'test_small': [sample for sample in test_samples if sample['size'] == 'small'],
        'test_medium': [sample for sample in test_samples if sample['size'] == 'medium'],
        'test_large': [sample for sample in test_samples if sample['size'] == 'large'],
    }

    if YOLO_DATASET_DIR.exists():
        shutil.rmtree(YOLO_DATASET_DIR)

    for split_name, split_samples in export_groups.items():
        output_image_dir = YOLO_DATASET_DIR / 'images' / split_name
        output_label_dir = YOLO_DATASET_DIR / 'labels' / split_name
        output_image_dir.mkdir(parents=True, exist_ok=True)
        output_label_dir.mkdir(parents=True, exist_ok=True)

        for index, sample in enumerate(split_samples):
            safe_name = f"{sample['dataset']}_{sample['size']}_{index:06d}_{sample['image_path'].name}"
            image_link = output_image_dir / safe_name
            label_output = output_label_dir / (Path(safe_name).stem + '.txt')
            os.symlink(sample['image_path'], image_link)

            one_class_lines = []
            for line in sample['label_path'].read_text().splitlines():
                parts = line.split()
                if len(parts) >= 5:
                    one_class_lines.append('0 ' + ' '.join(parts[1:5]))
            label_output.write_text('\n'.join(one_class_lines))

        print(split_name, 'images:', len(split_samples))

    print('Prepared fixed split with image links at:', YOLO_DATASET_DIR)

required_dirs = [
    YOLO_DATASET_DIR / folder / split
    for folder in ('images', 'labels')
    for split in SPLIT_NAMES
]
missing = [str(path) for path in required_dirs if not path.exists()]
if missing:
    raise FileNotFoundError('Missing expected split folders:\n' + '\n'.join(missing))
print('All fixed image and label splits are ready.')

In [ ]:
from PIL import Image

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
SPLITS = ('train', 'val', 'test', 'test_small', 'test_medium', 'test_large')

def yolo_box_to_coco(parts, image_width, image_height):
    if len(parts) != 5:
        return None
    _, cx, cy, bw, bh = map(float, parts)
    x = (cx - bw / 2) * image_width
    y = (cy - bh / 2) * image_height
    w = bw * image_width
    h = bh * image_height

    x1 = max(0.0, min(x, image_width))
    y1 = max(0.0, min(y, image_height))
    x2 = max(0.0, min(x + w, image_width))
    y2 = max(0.0, min(y + h, image_height))
    w = x2 - x1
    h = y2 - y1
    if w <= 0 or h <= 0:
        return None
    return [round(x1, 4), round(y1, 4), round(w, 4), round(h, 4)]

def convert_split_to_coco(split_name):
    image_dir = YOLO_DATASET_DIR / 'images' / split_name
    label_dir = YOLO_DATASET_DIR / 'labels' / split_name
    images = []
    annotations = []
    annotation_id = 1
    skipped_boxes = 0

    image_paths = sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTS)
    for image_id, image_path in enumerate(image_paths, start=1):
        with Image.open(image_path) as image:
            image_width, image_height = image.size

        images.append({
            'id': image_id,
            'file_name': image_path.name,
            'width': image_width,
            'height': image_height,
        })

        label_path = label_dir / f'{image_path.stem}.txt'
        if not label_path.exists():
            continue

        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            box = yolo_box_to_coco(parts, image_width, image_height)
            if box is None:
                skipped_boxes += 1
                continue
            annotations.append({
                'id': annotation_id,
                'image_id': image_id,
                'category_id': 0,
                'bbox': box,
                'area': round(box[2] * box[3], 4),
                'iscrowd': 0,
            })
            annotation_id += 1

    coco = {
        'info': {'description': 'Small defect detection: one-class D-FINE export'},
        'licenses': [],
        'images': images,
        'annotations': annotations,
        'categories': [{'id': 0, 'name': 'defect', 'supercategory': 'defect'}],
    }

    annotation_dir = COCO_ROOT / 'annotations'
    annotation_dir.mkdir(parents=True, exist_ok=True)
    output_path = annotation_dir / f'instances_{split_name}.json'
    output_path.write_text(json.dumps(coco, indent=2))

    return {
        'split': split_name,
        'images': len(images),
        'instances': len(annotations),
        'skipped_boxes': skipped_boxes,
        'annotation_file': str(output_path),
    }

conversion_rows = [convert_split_to_coco(split) for split in SPLITS]
for row in conversion_rows:
    print(row)

assert next(row['images'] for row in conversion_rows if row['split'] == 'train') > 0
assert next(row['images'] for row in conversion_rows if row['split'] == 'val') > 0
print('COCO annotations written to:', COCO_ROOT / 'annotations')

In [ ]:
# Write a small overlay config inside the D-FINE config folder.
# It inherits the official D-FINE-S custom configuration and changes only our dataset/run settings.
CONFIG_DIR = DFINE_REPO / 'configs' / 'dfine' / 'custom'
TRAIN_CONFIG = CONFIG_DIR / 'dfine_defect_s.yml'

def config_text(image_split, annotation_split, output_dir, epochs=EPOCHS):
    image_dir = YOLO_DATASET_DIR / 'images' / image_split
    ann_file = COCO_ROOT / 'annotations' / f'instances_{annotation_split}.json'
    return f'''__include__: [ './dfine_hgnetv2_s_custom.yml' ]

output_dir: {output_dir}
num_classes: 1
remap_mscoco_category: False
epochs: {epochs}

optimizer:
  type: AdamW
  lr: {BASE_LR}
  betas: [0.9, 0.999]
  weight_decay: 0.0001
  params:
    - params: '^(?=.*backbone)(?!.*norm|bn).*$'
      lr: {BACKBONE_LR}
    - params: '^(?=.*backbone)(?=.*norm|bn).*$'
      lr: {BACKBONE_LR}
      weight_decay: 0.0
    - params: '^(?=.*(?:encoder|decoder))(?=.*(?:norm|bn|bias)).*$'
      weight_decay: 0.0

train_dataloader:
  total_batch_size: {BATCH_SIZE}
  num_workers: {NUM_WORKERS}
  dataset:
    img_folder: {YOLO_DATASET_DIR / 'images' / 'train'}
    ann_file: {COCO_ROOT / 'annotations' / 'instances_train.json'}

val_dataloader:
  total_batch_size: {BATCH_SIZE}
  num_workers: {NUM_WORKERS}
  dataset:
    img_folder: {image_dir}
    ann_file: {ann_file}
'''

TRAIN_CONFIG.write_text(config_text('val', 'val', RUN_DIR))
print(TRAIN_CONFIG.read_text())

In [ ]:
# Train D-FINE-S. Checkpoint files are written under RUN_DIR and retained by Kaggle output.
train_command = [
    sys.executable, 'train.py', '-c', str(TRAIN_CONFIG),
    '--use-amp', '--seed', str(SEED)
]
print('Running:', ' '.join(train_command))
subprocess.run(train_command, cwd=DFINE_REPO, check=True)

In [ ]:
# Find the best checkpoint automatically.
checkpoint_candidates = sorted(RUN_DIR.rglob('*.pth'), key=lambda path: path.stat().st_mtime, reverse=True)
if not checkpoint_candidates:
    raise FileNotFoundError(f'No .pth checkpoint was found under {RUN_DIR}')

best_named = [path for path in checkpoint_candidates if 'best' in path.name.lower()]
BEST_CHECKPOINT = best_named[0] if best_named else checkpoint_candidates[0]
print('Available checkpoints:')
for path in checkpoint_candidates:
    print(path)
print('Using for all evaluations:', BEST_CHECKPOINT)

In [ ]:
# Evaluate the same checkpoint on validation and each fixed test subset.
# D-FINE uses val_dataloader for --test-only, so each config redirects that loader.
evaluation_sets = {
    'val': ('val', 'val'),
    'test_overall': ('test', 'test'),
    'test_small': ('test_small', 'test_small'),
    'test_medium': ('test_medium', 'test_medium'),
    'test_large': ('test_large', 'test_large'),
}

EVAL_LOG_DIR = FINAL_OUTPUT_DIR / 'evaluation_logs'
EVAL_LOG_DIR.mkdir(parents=True, exist_ok=True)

for eval_name, (image_split, annotation_split) in evaluation_sets.items():
    eval_config = CONFIG_DIR / f'dfine_defect_s_{eval_name}.yml'
    eval_output_dir = WORKING_ROOT / 'dfine_evaluations' / RUN_NAME / eval_name
    eval_config.write_text(config_text(image_split, annotation_split, eval_output_dir, epochs=EPOCHS))

    command = [
        sys.executable, 'train.py', '-c', str(eval_config),
        '--test-only', '-r', str(BEST_CHECKPOINT)
    ]
    print(f'\nRunning {eval_name}:', ' '.join(command))
    completed = subprocess.run(
        command, cwd=DFINE_REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    log_path = EVAL_LOG_DIR / f'{eval_name}.log'
    log_path.write_text(completed.stdout)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'{eval_name} evaluation failed. Read {log_path}')

print('All validation and test evaluations completed.')

In [ ]:
# Save the reproducibility artefacts in one Kaggle output folder.
import csv

FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(TRAIN_CONFIG, FINAL_OUTPUT_DIR / TRAIN_CONFIG.name)
shutil.copy2(BEST_CHECKPOINT, FINAL_OUTPUT_DIR / 'best_checkpoint.pth')
shutil.copytree(COCO_ROOT / 'annotations', FINAL_OUTPUT_DIR / 'coco_annotations', dirs_exist_ok=True)

with open(FINAL_OUTPUT_DIR / 'split_counts.csv', 'w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=conversion_rows[0].keys())
    writer.writeheader()
    writer.writerows(conversion_rows)

run_metadata = {
    'experiment': RUN_NAME,
    'model': 'D-FINE-S',
    'task': 'one-class defect detection',
    'image_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'seed': SEED,
    'training_split': 'train (all size buckets)',
    'validation_split': 'val (all size buckets)',
    'test_splits': ['test', 'test_small', 'test_medium', 'test_large'],
    'checkpoint': str(BEST_CHECKPOINT),
}
(FINAL_OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2))

print('Saved Kaggle output artefacts to:', FINAL_OUTPUT_DIR)
print('Evaluation logs:', EVAL_LOG_DIR)
print('Split counts:')
for row in conversion_rows:
    print(row['split'], 'images=', row['images'], 'instances=', row['instances'])